# JEPA Architecture Explainer (CIFAR-10)

A visual walk-through of the **current** JEPA architecture used in `JEPA_CIFAR10.ipynb` (v7).
Every component is shown end-to-end on a single CIFAR-10 image so you can trace
what the model actually sees and produces.

**Outline**
1. Config (same constants as training)
2. Data formation — image → 40% **pool** → 20% **context** + per-image **target coords**
3. Input encoding — RGB + Gaussian Fourier features
4. The two encoders — context (trained) vs. target (EMA)
5. Inside the encoder — `latent_pos` grid + **Gaussian attention bias**
6. Target side — **deterministic Gaussian soft-pool** of `g_tgt` over `latent_pos`
7. Predictor side — `proj_g(g) ⊕ (proj_query(γ(coord)) + mask_token)` → cross-attn
8. End-to-end forward — `g_ctx`, `h_pred`, `h_tgt`, per-coord loss + cosine
9. Full architecture schematic + tensor-shape table

> Notation: `B` = batch, `N_pix = 32×32 = 1024`, `K_pool = 410` (40%), `K_half = 205` (20%),
> `N_lat = 256`, `N_tgt = 64`, `D_emb = 512`, `D_pred = 256`, `D_pos = 192`.


## 1. Config — identical to `JEPA_CIFAR10.ipynb`

In [ ]:
import os, math, ssl, copy
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from einops import rearrange, repeat
import torchvision
from torchvision import transforms
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, FancyArrowPatch
from matplotlib.lines import Line2D

ssl._create_default_https_context = ssl._create_unverified_context
torch.manual_seed(0); np.random.seed(0)

# --- exactly matches JEPA_CIFAR10.ipynb v7 ---
IMAGE_SIZE  = 32
N_PIX       = IMAGE_SIZE * IMAGE_SIZE     # 1024
FRAC_POOL   = 0.40
K_POOL      = int(round(FRAC_POOL * N_PIX))   # 410
K_HALF      = K_POOL // 2                     # 205

CHANNELS    = 3
FOURIER_DIM = 96
POS_DIM     = FOURIER_DIM * 2                 # 192
INPUT_DIM   = CHANNELS + POS_DIM              # 195

LATENT_DIMS = (256, 384, 512)
NUM_LATENTS = (256, 256, 256)
EMBED_DIM   = LATENT_DIMS[-1]                 # 512
PRED_DIM    = 256
PRED_LAYERS = 4
PRED_HEADS  = 8
PRED_HEAD_D = 32

POS_BIAS_SIGMA = 0.5     # encoder Gaussian attention bias σ
POS_POOL_SIGMA = 0.15    # target-side Gaussian soft-pool σ
N_TGT          = 64      # per-image target coords sampled each iteration

DEVICE = "cpu"           # single-image walkthrough; CPU is enough
print(f"K_POOL={K_POOL}  K_HALF={K_HALF}  N_lat={NUM_LATENTS[0]}  D_emb={EMBED_DIM}  D_pred={PRED_DIM}")
print(f"σ_bias={POS_BIAS_SIGMA}  σ_pool={POS_POOL_SIGMA}  N_tgt={N_TGT}")


## 2. Data formation — pool, context, target

For each image:
- A **fixed 40 % pool** of pixel coordinates is drawn *once per image* with a seeded RNG
  (so it never changes across epochs).
- At train time, an **anchor** is picked uniformly inside the pool, and the **205 nearest pool
  pixels** to that anchor form one **context chunk** (`K_half = 205` pixels, the 20 % visible to the model).
- Independently, **`N_tgt = 64` target coordinates** are sampled uniformly from the pool —
  these are the positions where the predictor will try to match the target encoder.

Below we draw a real CIFAR-10 image and show all three views.


In [ ]:
# Load one CIFAR-10 image
transform = transforms.Compose([transforms.ToTensor(),
                                transforms.Normalize((0.5,)*3, (0.5,)*3)])
cifar = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
img, label = cifar[7]  # arbitrary sample
CLASSES = ['airplane','car','bird','cat','deer','dog','frog','horse','ship','truck']
print(f"Sample label: {CLASSES[label]}  shape={tuple(img.shape)}")

# Build the (y,x) coord grid in [-1, 1]², row-major
ys, xs = torch.meshgrid(
    torch.linspace(-1.0, 1.0, IMAGE_SIZE),
    torch.linspace(-1.0, 1.0, IMAGE_SIZE),
    indexing='ij',
)
coords_all = torch.stack([ys, xs], dim=-1).view(N_PIX, 2)
pix_all    = img.permute(1, 2, 0).reshape(-1, 3)
print(f"coords_all: {tuple(coords_all.shape)}    pix_all: {tuple(pix_all.shape)}")

# Sample the 40% pool (fixed per image — uses image-specific seed in the real loader)
rng       = np.random.RandomState(12345 + 7)   # 12345 = POOL_SEED, +7 = image idx
pool_idx  = rng.permutation(N_PIX)[:K_POOL]
pool_xy   = coords_all[pool_idx]

# Pick an anchor inside the pool, take K_half nearest pool members as context
anchor_local = rng.randint(K_POOL)
d2           = (pool_xy - pool_xy[anchor_local]).pow(2).sum(-1).numpy()
order        = np.argsort(d2, kind="stable")
ctx_local    = order[:K_HALF]
ctx_idx      = pool_idx[ctx_local]
ctx_xy       = coords_all[ctx_idx]

# Sample N_tgt target coords uniformly from the pool
tgt_local    = rng.choice(K_POOL, size=N_TGT, replace=False)
tgt_idx      = pool_idx[tgt_local]
tgt_xy       = coords_all[tgt_idx]

print(f"pool_idx: {pool_idx.shape}   ctx_idx: {ctx_idx.shape}   tgt_idx: {tgt_idx.shape}")
print(f"anchor coord = {pool_xy[anchor_local].tolist()}")


In [ ]:
# 5-panel data visualization
def show_view(ax, mask_idx, title, color='red', s=20, draw_anchor=False):
    im = ((img.permute(1,2,0) * 0.5) + 0.5).clamp(0,1).numpy()
    canvas = np.ones_like(im) * 0.85
    if mask_idx is not None:
        flat = canvas.reshape(-1, 3)
        flat[mask_idx] = im.reshape(-1, 3)[mask_idx]
    ax.imshow(canvas, extent=(-1, 1, 1, -1))   # y-axis flipped for image
    if draw_anchor:
        a = pool_xy[anchor_local]
        ax.scatter([a[1]], [a[0]], s=180, marker='*', c='yellow',
                   edgecolors='black', linewidths=1.2, zorder=5, label='anchor')
        ax.legend(loc='lower right', fontsize=7)
    ax.set_title(title, fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])

def show_coords(ax, xy, title, color='red', s=14):
    ax.imshow(((img.permute(1,2,0)*0.5)+0.5).clamp(0,1).numpy(), extent=(-1,1,1,-1), alpha=0.35)
    ax.scatter(xy[:,1].numpy(), xy[:,0].numpy(), s=s, c=color, edgecolors='black', linewidths=0.3)
    ax.set_title(title, fontsize=10)
    ax.set_xlim(-1.05, 1.05); ax.set_ylim(1.05, -1.05)
    ax.set_xticks([]); ax.set_yticks([])

fig, axes = plt.subplots(1, 5, figsize=(18, 4))
show_view(axes[0], None,    f"(a) original\n{CLASSES[label]}")
# Reveal the full image by passing all pixel indices
axes[0].imshow(((img.permute(1,2,0)*0.5)+0.5).clamp(0,1).numpy(), extent=(-1,1,1,-1))
axes[0].set_title(f"(a) original image\nclass = {CLASSES[label]}", fontsize=10)
show_view(axes[1], pool_idx, f"(b) 40% pool\n(K_pool = {K_POOL})", draw_anchor=True)
show_view(axes[2], ctx_idx,  f"(c) 20% context\n(K_half = {K_HALF})  → encoder input", draw_anchor=True)
show_coords(axes[3], tgt_xy, f"(d) target coords\n(N_tgt = {N_TGT})", color='lime', s=22)
# Overlay context + target on same plot
axes[4].imshow(((img.permute(1,2,0)*0.5)+0.5).clamp(0,1).numpy(), extent=(-1,1,1,-1), alpha=0.3)
axes[4].scatter(ctx_xy[:,1], ctx_xy[:,0], s=8, c='red',  alpha=0.6, label=f'context ({K_HALF})')
axes[4].scatter(tgt_xy[:,1], tgt_xy[:,0], s=18, c='lime', edgecolors='black', linewidths=0.4, label=f'target ({N_TGT})')
axes[4].set_xlim(-1.05, 1.05); axes[4].set_ylim(1.05, -1.05)
axes[4].set_xticks([]); axes[4].set_yticks([])
axes[4].set_title("(e) context + target overlay", fontsize=10)
axes[4].legend(loc='lower right', fontsize=7)

plt.suptitle("Data formation: each image gets fixed 40% pool → 20% context chunk + 64 target coords",
             y=1.02, fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()


## 3. Input encoding — per-pixel `[RGB ⊕ γ(y,x)]`

The encoder doesn't take the image directly. Each visible pixel becomes a 195-dim
token:

```
input_token[n]  =  [ R, G, B,    γ(y_n, x_n) ]
                    └──3──┘     └────192─────┘
                     pixel       Gaussian Fourier features
```

where `γ(y, x) = [sin(B·c), cos(B·c)]` with `B ∈ ℝ²ˣ⁹⁶` fixed-random
(`scale=15`). This gives the encoder a *positional* sense of where each
visible pixel lies, even though pixels can come from anywhere in the image.


In [ ]:
class GaussianFourierFeatures(nn.Module):
    def __init__(self, in_features, mapping_size, scale=15.0):
        super().__init__()
        self.register_buffer('B', torch.randn((in_features, mapping_size)) * scale)
    def forward(self, coords):
        proj = coords @ self.B
        return torch.cat([torch.sin(proj), torch.cos(proj)], dim=-1)

torch.manual_seed(0)
fourier = GaussianFourierFeatures(2, FOURIER_DIM, scale=15.0)
ctx_pix = pix_all[ctx_idx]                      # (205, 3)
ctx_pos = fourier(ctx_xy)                       # (205, 192)
ctx_tok = torch.cat([ctx_pix, ctx_pos], dim=-1) # (205, 195)
print(f"ctx_pixels: {tuple(ctx_pix.shape)}     ctx_pos: {tuple(ctx_pos.shape)}     ctx_token: {tuple(ctx_tok.shape)}")
print(f"token stats: mean={ctx_tok.mean():.3f}  std={ctx_tok.std():.3f}  min={ctx_tok.min():.2f}  max={ctx_tok.max():.2f}")

# Visualize: pixels, positional features, and concatenated tokens
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].imshow(ctx_pix.numpy(), aspect='auto', cmap='RdBu_r', vmin=-1, vmax=1)
axes[0].set_title("RGB component  (205 × 3)", fontsize=10)
axes[0].set_xlabel("channel (R,G,B)"); axes[0].set_ylabel("context pixel #")
axes[0].set_xticks([0,1,2]); axes[0].set_xticklabels(['R','G','B'])

axes[1].imshow(ctx_pos.numpy(), aspect='auto', cmap='RdBu_r', vmin=-1, vmax=1)
axes[1].set_title("γ(y,x) Gaussian Fourier features  (205 × 192)", fontsize=10)
axes[1].set_xlabel("feature dim"); axes[1].set_ylabel("context pixel #")

axes[2].imshow(ctx_tok.numpy(), aspect='auto', cmap='RdBu_r', vmin=-1, vmax=1)
axes[2].set_title("concatenated input token  (205 × 195)", fontsize=10)
axes[2].set_xlabel("dim (0-2: RGB | 3-194: γ)"); axes[2].set_ylabel("context pixel #")
axes[2].axvline(2.5, color='black', lw=1.5, ls='--')

plt.suptitle("Per-pixel input tokens: `data[n] = [R, G, B, γ(y_n, x_n)]`", fontweight='bold', y=1.03)
plt.tight_layout(); plt.show()


## 4. The two encoders — context (trained) vs. target (EMA)

JEPA uses a **siamese pair**:

| Encoder         | Sees what?                           | Weights              | Gradient? |
|-----------------|--------------------------------------|----------------------|-----------|
| `context_encoder` | 20 % chunk (`K_half = 205` pixels)   | Trained              | yes       |
| `target_encoder`  | **40 % pool** (all `K_pool = 410`)   | EMA of context (m≈0.999→1.0) | no        |

Both encoders have **identical architecture** (the OmniField cascade), and both
expose the same `latent_pos` parameter — a learnable 2-D anchor per latent
token. The target encoder's `latent_pos` slowly tracks the context encoder's
via EMA.

> Why does the target see *more*? Because we want it to provide a richer,
> more spatially-consistent target distribution than what the context view
> alone could give. The 40 % pool is the maximum we ever expose during
> training (still well short of full image).


## 5. Inside the encoder — `latent_pos` + Gaussian attention bias

The encoder follows OmniField's **ICMR cascade**: 3 perceiver-style blocks
that contract the variable-length pixel set into a fixed set of **256 latent
tokens**, each anchored to a learnable 2-D position.

### 5a. The `latent_pos` grid

At init, `latent_pos` is a **16×16 uniform grid** in `[-1, 1]²` with tiny
jitter. The grid is what gives the encoder its spatial inductive bias —
without it, all latents would be content-only and there'd be no way to ask
"what's at this coordinate?".


In [ ]:
# Build the latent_pos grid exactly as JEPAEncoder does at init
N_lat = NUM_LATENTS[0]   # 256
gs    = int(math.ceil(math.sqrt(N_lat)))   # 16
ys2   = torch.linspace(-1.0, 1.0, gs)
xs2   = torch.linspace(-1.0, 1.0, gs)
grid_y = ys2.unsqueeze(1).expand(gs, gs).contiguous()
grid_x = xs2.unsqueeze(0).expand(gs, gs).contiguous()
latent_pos = torch.stack([grid_y, grid_x], dim=-1).reshape(-1, 2)[:N_lat]
latent_pos = latent_pos + 0.01 * torch.randn_like(latent_pos)
print(f"latent_pos: {tuple(latent_pos.shape)}  var/axis={latent_pos.var(dim=0).tolist()}")

fig, axes = plt.subplots(1, 2, figsize=(12, 5.5))

# left: latent_pos grid overlaid on the image
axes[0].imshow(((img.permute(1,2,0)*0.5)+0.5).clamp(0,1).numpy(), extent=(-1,1,1,-1), alpha=0.45)
axes[0].scatter(latent_pos[:,1].numpy(), latent_pos[:,0].numpy(),
                s=22, c='cyan', edgecolors='black', linewidths=0.4, label=f'latent_pos ({N_lat})')
axes[0].set_xlim(-1.05, 1.05); axes[0].set_ylim(1.05, -1.05)
axes[0].set_title(f"`latent_pos` at init: 16×16 grid in [-1,1]²\n(each cyan dot is one of {N_lat} latent tokens)", fontsize=10)
axes[0].legend(loc='lower right', fontsize=8)
axes[0].set_xticks([]); axes[0].set_yticks([])

# right: just the grid in coordinate space, color = index
axes[1].scatter(latent_pos[:,1].numpy(), latent_pos[:,0].numpy(),
                s=40, c=np.arange(N_lat), cmap='viridis', edgecolors='black', linewidths=0.3)
axes[1].set_xlim(-1.05, 1.05); axes[1].set_ylim(1.05, -1.05)
axes[1].set_xlabel("x"); axes[1].set_ylabel("y")
axes[1].set_title("latent_pos in coord space\n(color = latent index 0…255)", fontsize=10)
axes[1].grid(alpha=0.3)

plt.tight_layout(); plt.show()


### 5b. Gaussian attention bias

Inside each cascaded block's cross-attention, latent `l` attends to input
pixel `n` with logits:

$$
\text{sim}(l, n) \;=\; \frac{q_l \cdot k_n}{\sqrt{d}} \;-\; \frac{\|\text{latent\_pos}[l] - \text{coord}[n]\|^2}{2\sigma^2}, \quad \sigma = 0.5
$$

The bias term is **negative-distance** in coord units, so latents preferentially
attend to inputs **near their anchor**. Below: heatmaps of the bias for 3 chosen
latents, overlaid on the context pixels.


In [ ]:
# Visualize the Gaussian attention bias for 3 chosen latents
sigma = POS_BIAS_SIGMA
chosen = [0, 128, 245]  # corners + middle

# d²[l, n] for chosen l × all input coords
d2 = (latent_pos[chosen].unsqueeze(1) - ctx_xy.unsqueeze(0)).pow(2).sum(-1)   # (3, 205)
bias = -d2 / (2.0 * sigma**2)                                                  # (3, 205)
attn_no_bias = torch.softmax(torch.zeros_like(bias), dim=-1)
attn_bias    = torch.softmax(bias, dim=-1)

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for col, l in enumerate(chosen):
    # Row 0: attention weights without bias (uniform)
    ax = axes[0, col]
    ax.imshow(((img.permute(1,2,0)*0.5)+0.5).clamp(0,1).numpy(), extent=(-1,1,1,-1), alpha=0.3)
    sc = ax.scatter(ctx_xy[:,1], ctx_xy[:,0], s=20,
                    c=attn_no_bias[col].numpy(), cmap='hot', vmin=0, vmax=attn_bias[col].max()*1.1)
    a = latent_pos[l]
    ax.scatter([a[1]], [a[0]], s=300, marker='*', c='cyan', edgecolors='black',
               linewidths=1.5, zorder=5)
    ax.set_xlim(-1.05,1.05); ax.set_ylim(1.05,-1.05); ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(f"latent {l} — without bias\n(uniform 1/{K_HALF})", fontsize=9)
    plt.colorbar(sc, ax=ax, fraction=0.04)

    # Row 1: attention weights with Gaussian bias
    ax = axes[1, col]
    ax.imshow(((img.permute(1,2,0)*0.5)+0.5).clamp(0,1).numpy(), extent=(-1,1,1,-1), alpha=0.3)
    sc = ax.scatter(ctx_xy[:,1], ctx_xy[:,0], s=20,
                    c=attn_bias[col].numpy(), cmap='hot')
    ax.scatter([a[1]], [a[0]], s=300, marker='*', c='cyan', edgecolors='black',
               linewidths=1.5, zorder=5)
    ax.set_xlim(-1.05,1.05); ax.set_ylim(1.05,-1.05); ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(f"latent {l} — softmax(−d²/2σ²), σ={sigma}", fontsize=9)
    plt.colorbar(sc, ax=ax, fraction=0.04)

plt.suptitle("Gaussian attention bias: each latent sees a soft Gaussian neighborhood around its anchor (★)",
             fontweight='bold', y=1.01)
plt.tight_layout(); plt.show()


## 6. Target side — deterministic Gaussian soft-pool

The target features at each query coordinate `c_q` are a **parameter-free**
position-weighted average of the target encoder's latent set `g_tgt`:

$$
w_{q,l} = \mathrm{softmax}_l\!\left(-\frac{\|c_q - \text{latent\_pos}[l]\|^2}{2\sigma_\text{pool}^2}\right), \quad \sigma_\text{pool} = 0.15
$$

$$
h_\text{tgt}[q] \;=\; \sum_l w_{q,l}\; g_\text{tgt}[l]
$$

Crucially **no learnable parameters on the target side** — this was the v5 fix
that killed a trivial-mirror collapse from earlier versions. Each target
coordinate just averages the latents close to it.


In [ ]:
# Visualize soft-pool weights for 3 different target coords
sigma_pool = POS_POOL_SIGMA
example_q = tgt_xy[[0, 30, 60]]      # 3 of the 64 target coords

d2 = (example_q.unsqueeze(1) - latent_pos.unsqueeze(0)).pow(2).sum(-1)  # (3, 256)
w  = torch.softmax(-d2 / (2.0 * sigma_pool**2), dim=-1)                 # (3, 256)

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for col in range(3):
    # Top: spatial overlay of weight on latent_pos
    ax = axes[0, col]
    ax.imshow(((img.permute(1,2,0)*0.5)+0.5).clamp(0,1).numpy(), extent=(-1,1,1,-1), alpha=0.3)
    sc = ax.scatter(latent_pos[:,1], latent_pos[:,0], s=30,
                    c=w[col].numpy(), cmap='viridis', edgecolors='black', linewidths=0.2)
    q = example_q[col]
    ax.scatter([q[1]], [q[0]], s=400, marker='X', c='red', edgecolors='black', linewidths=1.5, zorder=5,
               label='target coord')
    ax.set_xlim(-1.05,1.05); ax.set_ylim(1.05,-1.05); ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(f"target coord #{col}  (y={q[0]:.2f}, x={q[1]:.2f})\n"
                 f"weight over 256 latents (σ={sigma_pool})", fontsize=9)
    ax.legend(loc='lower right', fontsize=7)
    plt.colorbar(sc, ax=ax, fraction=0.04)

    # Bottom: histogram of weights (how peaked the soft-pool is)
    ax = axes[1, col]
    ws = sorted(w[col].numpy(), reverse=True)
    ax.bar(range(20), ws[:20], color='steelblue', edgecolor='black')
    ax.set_xlabel("latent rank (top 20)"); ax.set_ylabel("weight")
    eff_k = 1.0 / (w[col]**2).sum().item()
    ax.set_title(f"top-20 weights  |  effective #latents = {eff_k:.1f}", fontsize=9)
    ax.grid(alpha=0.3)

plt.suptitle("Deterministic Gaussian soft-pool: each target coord ≈ weighted avg of nearby latents",
             fontweight='bold', y=1.01)
plt.tight_layout(); plt.show()


## 7. Predictor side — `CoordReadout`

The predictor is a **small transformer** (4 layers, 8 heads, dim 256) that
takes:

- **Context tokens**: `proj_g(g_ctx)` ∈ ℝ²⁵⁶×²⁵⁶ — pure content from the encoder
  (no positional pathway — v5 removed it; position must come from `g_ctx`'s
  *content* which was made position-aware by the Gaussian attention bias).
- **Target tokens**: `proj_query(γ(c_q)) + mask_token` ∈ ℝᴺᵗᵍᵗ×²⁵⁶ — one per
  target coordinate, asking *"what should `h_tgt` be here?"*

They're concatenated, run through self-attention + FFN blocks, then the
target-token slice is projected back to `D_emb = 512` and compared to `h_tgt`
via smooth-L1.

```
  ┌─ proj_g(g_ctx) ─────────────────┐     (B, 256, 256)
  │                                 │
  │     [   self-attn × 4   ]       │  →  take last N_tgt tokens
  │                                 │
  └─ proj_query(γ(c_q)) + mask ─────┘     (B,  64, 256)
```


## 8. Full model definitions (copied verbatim from training)

In [ ]:
def get_sinusoidal_embeddings(n, d):
    position = torch.arange(n, dtype=torch.float).unsqueeze(1)
    div_term = torch.exp(torch.arange(0, d, 2).float() * -(math.log(10000.0) / d))
    pe = torch.zeros(n, d)
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe

class PreNorm(nn.Module):
    def __init__(self, dim, fn, context_dim=None):
        super().__init__()
        self.fn = fn
        self.norm = nn.LayerNorm(dim)
        self.norm_context = nn.LayerNorm(context_dim) if context_dim is not None else None
    def forward(self, x, **kw):
        x = self.norm(x)
        if self.norm_context is not None:
            kw['context'] = self.norm_context(kw['context'])
        return self.fn(x, **kw)

class GEGLU(nn.Module):
    def forward(self, x):
        x, gates = x.chunk(2, dim=-1)
        return x * F.gelu(gates)

class FeedForward(nn.Module):
    def __init__(self, dim, mult=4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, dim * mult * 2), GEGLU(),
            nn.Linear(dim * mult, dim),
        )
    def forward(self, x): return self.net(x)

class Attention(nn.Module):
    def __init__(self, query_dim, context_dim=None, heads=8, dim_head=64):
        super().__init__()
        inner = dim_head * heads
        context_dim = context_dim if context_dim is not None else query_dim
        self.scale, self.heads = dim_head ** -0.5, heads
        self.to_q   = nn.Linear(query_dim, inner, bias=False)
        self.to_kv  = nn.Linear(context_dim, inner * 2, bias=False)
        self.to_out = nn.Linear(inner, query_dim)
    def forward(self, x, context=None, mask=None, attn_bias=None):
        h = self.heads
        q = self.to_q(x)
        context = context if context is not None else x
        k, v = self.to_kv(context).chunk(2, dim=-1)
        q, k, v = map(lambda t: rearrange(t, 'b n (h d) -> (b h) n d', h=h), (q, k, v))
        sim = torch.einsum('b i d, b j d -> b i j', q, k) * self.scale
        if attn_bias is not None:
            B = x.shape[0]; H = sim.shape[0] // B
            ab = attn_bias.unsqueeze(1).expand(-1, H, -1, -1).reshape(B*H, attn_bias.shape[1], attn_bias.shape[2])
            sim = sim + ab
        attn = sim.softmax(dim=-1)
        out  = torch.einsum('b i j, b j d -> b i d', attn, v)
        return self.to_out(rearrange(out, '(b h) n d -> b n (h d)', h=h))

class CascadedBlock(nn.Module):
    def __init__(self, dim, n_latents, input_dim, cross_heads, cross_dim_head,
                 self_heads, self_dim_head, residual_dim=None, pos_bias_sigma=0.5):
        super().__init__()
        self.latents = nn.Parameter(get_sinusoidal_embeddings(n_latents, dim), requires_grad=True)
        self.cross_attn = PreNorm(dim, Attention(dim, input_dim, heads=cross_heads, dim_head=cross_dim_head),
                                  context_dim=input_dim)
        self.self_attn  = PreNorm(dim, Attention(dim, heads=self_heads, dim_head=self_dim_head))
        self.residual_proj = nn.Linear(residual_dim, dim) if residual_dim and residual_dim != dim else None
        self.ff = PreNorm(dim, FeedForward(dim))
        self.pos_bias_sigma = pos_bias_sigma
    def forward(self, context, residual=None, latent_pos=None, input_coords=None):
        b = context.size(0)
        x = repeat(self.latents, 'n d -> b n d', b=b)
        attn_bias = None
        if latent_pos is not None and input_coords is not None:
            d2 = (latent_pos.unsqueeze(0).unsqueeze(2) - input_coords.unsqueeze(1)).pow(2).sum(-1)
            attn_bias = -d2 / (2.0 * self.pos_bias_sigma**2)
        x = self.cross_attn(x, context=context, attn_bias=attn_bias) + x
        if residual is not None:
            r = self.residual_proj(residual) if self.residual_proj is not None else residual
            x = x + r
        x = self.self_attn(x) + x
        x = self.ff(x) + x
        return x

class JEPAEncoder(nn.Module):
    def __init__(self, input_dim=INPUT_DIM, latent_dims=LATENT_DIMS, num_latents=NUM_LATENTS,
                 cross_heads=4, cross_dim_head=128, self_heads=8, self_dim_head=128,
                 num_trunk_layers=4, pos_bias_sigma=POS_BIAS_SIGMA):
        super().__init__()
        self.encoder_blocks = nn.ModuleList()
        prev = None
        for d, n in zip(latent_dims, num_latents):
            self.encoder_blocks.append(
                CascadedBlock(d, n, input_dim, cross_heads, cross_dim_head,
                              self_heads, self_dim_head, residual_dim=prev,
                              pos_bias_sigma=pos_bias_sigma))
            prev = d
        final = latent_dims[-1]
        self.trunk = nn.ModuleList([
            nn.ModuleList([
                PreNorm(final, Attention(final, heads=self_heads, dim_head=self_dim_head)),
                PreNorm(final, FeedForward(final)),
            ]) for _ in range(num_trunk_layers)
        ])
        self.latent_dim = final
        N_lat = num_latents[0]
        gs = max(2, int(math.ceil(math.sqrt(N_lat))))
        ys = torch.linspace(-1.0, 1.0, gs); xs = torch.linspace(-1.0, 1.0, gs)
        gy = ys.unsqueeze(1).expand(gs, gs).contiguous()
        gx = xs.unsqueeze(0).expand(gs, gs).contiguous()
        grid = torch.stack([gy, gx], dim=-1).reshape(-1, 2)[:N_lat]
        grid = grid + 0.01 * torch.randn_like(grid)
        self.latent_pos = nn.Parameter(grid, requires_grad=True)
    def forward(self, data, input_coords):
        residual = None
        for block in self.encoder_blocks:
            residual = block(data, residual=residual,
                             latent_pos=self.latent_pos, input_coords=input_coords)
        for sa, ff in self.trunk:
            residual = sa(residual) + residual
            residual = ff(residual) + residual
        soft_pos = self.latent_pos.unsqueeze(0).expand(data.size(0), -1, -1)
        return residual, soft_pos

class CoordReadout(nn.Module):
    def __init__(self, latent_dim=EMBED_DIM, predictor_dim=PRED_DIM,
                 query_in_dim=POS_DIM, num_layers=PRED_LAYERS,
                 heads=PRED_HEADS, dim_head=PRED_HEAD_D):
        super().__init__()
        self.proj_g     = nn.Linear(latent_dim, predictor_dim)
        self.proj_query = nn.Linear(query_in_dim, predictor_dim)
        self.mask_token = nn.Parameter(torch.zeros(1, 1, predictor_dim))
        nn.init.trunc_normal_(self.mask_token, std=0.02)
        self.blocks = nn.ModuleList([
            nn.ModuleList([
                PreNorm(predictor_dim, Attention(predictor_dim, heads=heads, dim_head=dim_head)),
                PreNorm(predictor_dim, FeedForward(predictor_dim)),
            ]) for _ in range(num_layers)
        ])
        self.norm     = nn.LayerNorm(predictor_dim)
        self.proj_out = nn.Linear(predictor_dim, latent_dim)
    def forward(self, g, query_coords_enc):
        B, N_lat, _ = g.shape
        ctx_tok = self.proj_g(g)
        tgt_tok = self.proj_query(query_coords_enc) + self.mask_token
        x = torch.cat([ctx_tok, tgt_tok], dim=1)
        for sa, ff in self.blocks:
            x = sa(x) + x
            x = ff(x) + x
        x = self.norm(x)
        return self.proj_out(x[:, N_lat:])

def soft_pool_targets(g, latent_pos_p, query_coord, sigma):
    d2 = (query_coord.unsqueeze(2) - latent_pos_p.unsqueeze(0).unsqueeze(0)).pow(2).sum(-1)
    w  = F.softmax(-d2 / (2.0 * sigma**2), dim=-1)
    return torch.einsum('b q l, b l d -> b q d', w, g)

torch.manual_seed(0)
fourier_encoder = GaussianFourierFeatures(2, FOURIER_DIM, scale=15.0)
context_encoder = JEPAEncoder()
target_encoder  = copy.deepcopy(context_encoder)
predictor       = CoordReadout()
for p in target_encoder.parameters(): p.requires_grad_(False)

n_enc  = sum(p.numel() for p in context_encoder.parameters())
n_pred = sum(p.numel() for p in predictor.parameters())
print(f"context_encoder: {n_enc/1e6:.2f}M params")
print(f"predictor      : {n_pred/1e6:.2f}M params")
print(f"target_encoder : {n_enc/1e6:.2f}M params  (frozen, EMA-updated)")


## 9. End-to-end forward — context & target & predictor together

In [ ]:
# Encode inputs
def encode_input(pixels, coords):
    return torch.cat([pixels, fourier_encoder(coords)], dim=-1)
def encode_query(coords):
    return fourier_encoder(coords)

# Add batch dim of 1
ctx_p_b = pix_all[ctx_idx].unsqueeze(0)
ctx_c_b = ctx_xy.unsqueeze(0)
pool_p_b = pix_all[pool_idx].unsqueeze(0)
pool_c_b = coords_all[pool_idx].unsqueeze(0)
tgt_c_b  = tgt_xy.unsqueeze(0)

ctx_data  = encode_input(ctx_p_b,  ctx_c_b)
pool_data = encode_input(pool_p_b, pool_c_b)
tgt_q_enc = encode_query(tgt_c_b)

print(f"shapes entering the encoders:")
print(f"  ctx_data   : {tuple(ctx_data.shape)}    ← context_encoder input")
print(f"  pool_data  : {tuple(pool_data.shape)}    ← target_encoder input")
print(f"  tgt_q_enc  : {tuple(tgt_q_enc.shape)}    ← query coords for predictor")

with torch.no_grad():
    # Target side
    g_tgt, _   = target_encoder(pool_data, pool_c_b)
    h_tgt_raw  = soft_pool_targets(g_tgt, target_encoder.latent_pos, tgt_c_b, POS_POOL_SIGMA)
    h_tgt      = F.layer_norm(h_tgt_raw, (h_tgt_raw.size(-1),))
    # Context side
    g_ctx, _   = context_encoder(ctx_data, ctx_c_b)
    h_pred     = predictor(g_ctx, tgt_q_enc)
    loss       = F.smooth_l1_loss(h_pred, h_tgt)
    cos        = F.cosine_similarity(h_pred, h_tgt, dim=-1)

print(f"\nintermediate features:")
print(f"  g_tgt      : {tuple(g_tgt.shape)}      ← target encoder latent set")
print(f"  h_tgt_raw  : {tuple(h_tgt_raw.shape)}   ← soft-pool target features (pre-LayerNorm)")
print(f"  g_ctx      : {tuple(g_ctx.shape)}      ← context encoder latent set")
print(f"  h_pred     : {tuple(h_pred.shape)}     ← predictor output (matches h_tgt shape)")
print(f"\nloss = {loss.item():.4f}      cos(h_pred, h_tgt) = {cos.mean():.3f} ± {cos.std():.3f}")


In [ ]:
# Visualize: per-target-coord loss heat, per-target-coord cosine, h_pred & h_tgt heatmaps
fig = plt.figure(figsize=(16, 11))
gs = fig.add_gridspec(2, 3, hspace=0.32, wspace=0.28)

# (a) per-coord smooth-L1 loss laid on image
per_loss = F.smooth_l1_loss(h_pred[0], h_tgt[0], reduction='none').mean(-1).numpy()
ax = fig.add_subplot(gs[0, 0])
ax.imshow(((img.permute(1,2,0)*0.5)+0.5).clamp(0,1).numpy(), extent=(-1,1,1,-1), alpha=0.4)
sc = ax.scatter(tgt_xy[:,1], tgt_xy[:,0], s=80, c=per_loss, cmap='Reds',
                edgecolors='black', linewidths=0.4)
ax.set_xlim(-1.05,1.05); ax.set_ylim(1.05,-1.05); ax.set_xticks([]); ax.set_yticks([])
ax.set_title(f"(a) per-coord smooth-L1 loss\n(mean over 512 dims)  total={loss.item():.4f}", fontsize=10)
plt.colorbar(sc, ax=ax, fraction=0.04)

# (b) per-coord cosine
ax = fig.add_subplot(gs[0, 1])
ax.imshow(((img.permute(1,2,0)*0.5)+0.5).clamp(0,1).numpy(), extent=(-1,1,1,-1), alpha=0.4)
sc = ax.scatter(tgt_xy[:,1], tgt_xy[:,0], s=80, c=cos[0].numpy(), cmap='RdBu_r',
                vmin=-1, vmax=1, edgecolors='black', linewidths=0.4)
ax.set_xlim(-1.05,1.05); ax.set_ylim(1.05,-1.05); ax.set_xticks([]); ax.set_yticks([])
ax.set_title(f"(b) per-coord cos(h_pred, h_tgt)\nmean={cos.mean():.3f}", fontsize=10)
plt.colorbar(sc, ax=ax, fraction=0.04)

# (c) cosine histogram
ax = fig.add_subplot(gs[0, 2])
ax.hist(cos[0].numpy(), bins=20, color='steelblue', edgecolor='black')
ax.axvline(cos.mean().item(), color='red', ls='--', label=f'mean={cos.mean():.3f}')
ax.set_xlabel("cosine similarity"); ax.set_ylabel("count")
ax.set_xlim(-1, 1)
ax.set_title("(c) distribution of per-coord cosine\n(at init, untrained model)", fontsize=10)
ax.legend(fontsize=8); ax.grid(alpha=0.3)

# (d) h_pred matrix (N_tgt × D_emb), capped to first 64 dims for readability
ax = fig.add_subplot(gs[1, 0])
ax.imshow(h_pred[0, :, :64].numpy(), aspect='auto', cmap='RdBu_r')
ax.set_title("(d) h_pred[:, :64]  shape=(64, 512)", fontsize=10)
ax.set_xlabel("embed dim (0-63 shown)"); ax.set_ylabel("target coord")

# (e) h_tgt matrix
ax = fig.add_subplot(gs[1, 1])
ax.imshow(h_tgt[0, :, :64].numpy(), aspect='auto', cmap='RdBu_r')
ax.set_title("(e) h_tgt[:, :64]  shape=(64, 512)", fontsize=10)
ax.set_xlabel("embed dim (0-63 shown)"); ax.set_ylabel("target coord")

# (f) residual h_pred - h_tgt
ax = fig.add_subplot(gs[1, 2])
diff = (h_pred - h_tgt)[0, :, :64].numpy()
m = max(abs(diff.min()), abs(diff.max()))
ax.imshow(diff, aspect='auto', cmap='RdBu_r', vmin=-m, vmax=m)
ax.set_title("(f) residual = h_pred − h_tgt", fontsize=10)
ax.set_xlabel("embed dim (0-63 shown)"); ax.set_ylabel("target coord")

plt.suptitle("Forward pass on one CIFAR-10 image (untrained init): the JEPA objective at a glance",
             fontweight='bold', y=0.995)
plt.show()


## 10. Full architecture schematic

```
                              ┌──────────────────────────────────────────────┐
                              │           CIFAR-10 image (32×32×3)           │
                              └──────────────────────────────────────────────┘
                                       │            │            │
                                       │            │            │
                  fixed 40% pool ──────┘            │            │
                       (410 pixels)                 │            │
                              │                     │            │
                  ┌───────────┴───────────┐         │            │
                  │ 20% context (205 px)  │         │            │
                  │ 64 target coords      │         │            │
                  └─────┬─────────────┬───┘         │            │
                        │             │             │            │
                        │             │             │            │
   per-pixel tokens     │             │             │            │
   [R,G,B, γ(y,x)] ─────┼─────────────┼─────────────┘            │
        195-d           │             │                          │
                        │             │             pool tokens (410 × 195)
                        │             │                          │
                        ▼             ▼                          ▼
              ┌──────────────────┐  ┌─────────────────┐  ┌──────────────────┐
              │ context_encoder  │  │ target_encoder  │  │  pool input data │
              │  (trained,       │  │  (EMA of ctx,   │  └──────────────────┘
              │   3-stage ICMR + │  │   frozen)       │           │
              │   4-layer trunk) │  │   ICMR+trunk    │◄──────────┘
              │   + latent_pos   │  │   + latent_pos  │
              └────────┬─────────┘  └────────┬────────┘
                       │ g_ctx               │ g_tgt
                       │ (1, 256, 512)       │ (1, 256, 512)
                       │                     │
                       │                     │  deterministic Gaussian
                       │                     │  soft-pool (σ=0.15)
                       │                     │  over latent_pos
                       │                     ▼
                       │           ┌─────────────────────┐
                       │           │     h_tgt_raw       │
                       │           │     (1, 64, 512)    │
                       │           │   LayerNorm →h_tgt  │
                       │           └─────────────────────┘
                       │                     │
                       │                     │
                       ▼                     │
              ┌────────────────────┐         │
              │     predictor      │         │
              │  proj_g(g_ctx)     │         │
              │  ⊕                 │         │
              │  proj_query(γ(c_q))│         │
              │   + mask_token     │         │
              │  → 4-layer SA      │         │
              │  → proj_out        │         │
              └─────────┬──────────┘         │
                        │ h_pred             │
                        │ (1, 64, 512)       │
                        │                    │
                        └─────►  smooth_L1(h_pred, h_tgt)  ◄──────┘
                                       │
                                       └─►  back-prop to context_encoder + predictor only
                                            target_encoder updated via EMA from context
```

### Tensor-shape table

| Stage                 | Tensor       | Shape (single image)  | Notes                                  |
|-----------------------|--------------|-----------------------|----------------------------------------|
| pool pixels           | `pool_p`     | `(410, 3)`            | 40 % of 1024                           |
| pool coords           | `pool_c`     | `(410, 2)`            | in `[-1, 1]²`                          |
| context pixels        | `ctx_p`      | `(205, 3)`            | 20 % — what the trained encoder sees   |
| context coords        | `ctx_c`      | `(205, 2)`            |                                        |
| target coords         | `tgt_c`      | `(64, 2)`             | resampled per iteration from the pool  |
| input tokens          | `ctx_data`   | `(205, 195)`          | `[R,G,B, γ(y,x)]`, γ ∈ ℝ¹⁹²            |
| target enc input      | `pool_data`  | `(410, 195)`          |                                        |
| latent set (ctx)      | `g_ctx`      | `(256, 512)`          | ICMR cascade output                    |
| latent set (tgt)      | `g_tgt`      | `(256, 512)`          | from EMA target encoder                |
| target features (raw) | `h_tgt_raw`  | `(64, 512)`           | softmax-weighted avg of `g_tgt`        |
| target features       | `h_tgt`      | `(64, 512)`           | LayerNormed `h_tgt_raw` − DINO center  |
| predictor input       | `tgt_q_enc`  | `(64, 192)`           | `γ(tgt_c)`                             |
| prediction            | `h_pred`     | `(64, 512)`           | predictor output                       |
| loss                  | scalar       | `()`                  | smooth-L1(h_pred, h_tgt) ∈ ℝ           |

### Loss

$$
\mathcal{L} \;=\; \frac{1}{N_\text{tgt} \cdot D_\text{emb}} \sum_{q,d} \mathrm{smooth\_L1}\Big(h_\text{pred}[q,d],\; h_\text{tgt}[q,d]\Big)
$$

> No VICReg, no variance regularizer, no warmup — pure smooth-L1 distillation
> with EMA target + DINO centering, on a learnable spatial scaffold.


## 11. Why might this converge prematurely?

From the latest 300-epoch run we see:
- `Lj` plateaus around 0.1 – 0.19
- `cos(h_pred, h_tgt) ≈ 0.85` and stays there
- `|g|` keeps inflating (23 → 1000+) while `σt(g)` grows in lockstep
- Probe accuracy plateaus 26–29 % around epoch 20–60, no further gain

**Likely root cause: the target side is *too easy***.

The target is a softmax-weighted average of `g_tgt` over only the latents
nearest each target coord (σ_pool=0.15 ≈ 3 latents effective). The
predictor only needs to recover *the direction of the mean of the few
nearest latents* — it doesn't have to recover any fine-grained content,
and certainly doesn't have to predict anything for unseen pixels because
the *target coords are inside the pool the target encoder already saw*.

In short: at each target coord `c`, the target features are something the
target encoder *directly looked at*, while the predictor sees a different
subset (the context). Since the latents are shared by content, a fairly
trivial copy from neighbouring latents satisfies smooth-L1 to cos≈0.85,
and there's no further gradient.

**Things that might unlock more:**
1. Force target coords to be **outside the context chunk** (genuine prediction).
2. Sharper σ_pool so each target coord depends on **fewer latents**.
3. Larger `N_tgt` and/or a contrastive negative for `h_tgt`.
4. Asymmetric encoder/predictor depth — the predictor is currently very small.
